In [0]:
#We learn Autoloading of Incremental data from cloud source, Schema evolution, 
cloudsrc="/Volumes/prodcatalogue/prodschema/datalake/"#s3 storage path
#cloudsrc="gs://izsourcebucket/Master_City_List_hour1.csv"
bronzetgt="/Volumes/prodcatalogue/prodschema/bronze/"
#To resolve it, you must use a data source that is accessible from your AWS-based Databricks workspace, such as an S3 bucket or a Unity Catalog volume.
ckptlocation="/Volumes/prodcatalogue/prodschema/chkpt_location/"#stores the files copied information post write is successful
schemalocation="/Volumes/prodcatalogue/prodschema/schema_location/"#stores the inferred schema of the source data
df1=spark.readStream.format("cloudFiles")\
.option("cloudFiles.format","csv")\
.option("cloudFiles.maxFilesPerTrigger",1)\
.option("cloudFiles.inferColumnTypes",True)\
.option("cloudFiles.schemaEvolutionMode","addNewColumns")\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("header",True)\
.load(cloudsrc)#this can be s3/adls/gcs
#.option("cloudFiles.useNotifications", "true") (Remove this option to enable directory listing)
#maxFilesPerTrigger - this property help spark to process howmany files in an iteration to control the resource utilization (all files will be processed ultimately)
     

In [0]:
#realtime trigger is not possible in free serverless
#writeStream will read data from df1 (materialized here) and write to bronzetgt using the schema generated by reader and checkpoint info stored
df1.writeStream.trigger(availableNow=True)\
.option("checkpointLocation", ckptlocation)\
.option("cloudFiles.schemaLocation", schemalocation)\
.option("mergeSchema", "true") \
.start(bronzetgt)
#.option("mergeSchema", "true") \

In [0]:
spark.read.format("delta").load(bronzetgt).orderBy("shipment_id").show(100)